# 03 – Data Wrangling

**Goal:** understand the data, handle missing values, and create the training label **`Class`**
(1 = first stage landed successfully, 0 = it did not).

Input: `data/dataset_part_1.csv` (from notebook 01).
Output: `data/dataset_part_2.csv` (with `Class`) and `data/dataset_part_3.csv` (one-hot encoded features for machine learning).

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/dataset_part_1.csv")
print(df.shape)
df.head()

(90, 17)


,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude
0,1,2010-06-04,Falcon 9,6123.547647,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0003,-80.577366,28.561857
1,2,2012-05-22,Falcon 9,525.000000,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0005,-80.577366,28.561857
2,3,2013-03-01,Falcon 9,677.000000,ISS,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0007,-80.577366,28.561857
3,4,2013-09-29,Falcon 9,500.000000,PO,VAFB SLC 4E,False Ocean,1,False,False,False,NaN,1.0,0,B1003,-120.610829,34.632093
4,5,2013-12-03,Falcon 9,3170.000000,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1004,-80.577366,28.561857


## 1. Missing values

In [2]:
# percent of missing values in each column
(df.isnull().sum() / len(df) * 100).round(2)

FlightNumber       0.00
Date               0.00
BoosterVersion     0.00
PayloadMass        0.00
Orbit              0.00
LaunchSite         0.00
Outcome            0.00
Flights            0.00
GridFins           0.00
Reused             0.00
Legs               0.00
LandingPad        28.89
Block              0.00
ReusedCount        0.00
Serial             0.00
Longitude          0.00
Latitude           0.00
dtype: float64

* `PayloadMass` was already filled with the mean in notebook 01, so it has no missing values.
* `LandingPad` is missing when the booster did **not** use a landing pad (no landing attempt or ocean landing). We keep it as missing – it becomes an "all zeros" row after one-hot encoding.

In [3]:
df.dtypes

FlightNumber        int64
Date               object
BoosterVersion     object
PayloadMass       float64
Orbit              object
LaunchSite         object
Outcome            object
Flights             int64
GridFins             bool
Reused               bool
Legs                 bool
LandingPad         object
Block             float64
ReusedCount         int64
Serial             object
Longitude         float64
Latitude          float64
dtype: object

## 2. Number of launches per launch site

In [4]:
df["LaunchSite"].value_counts()

LaunchSite
CCAFS SLC 40    55
KSC LC 39A      22
VAFB SLC 4E     13
Name: count, dtype: int64

## 3. Number of launches per orbit

In [5]:
df["Orbit"].value_counts()

Orbit
GTO      27
ISS      21
VLEO     14
PO        9
LEO       7
SSO       5
MEO       3
HEO       1
ES-L1     1
SO        1
GEO       1
Name: count, dtype: int64

## 4. Landing outcomes
`Outcome` = "landing success" + "landing type":
* `True ASDS` = landed on a drone ship, `True RTLS` = landed on a ground pad, `True Ocean` = controlled landing in the ocean
* `False ...` = landing attempt failed, `None None` / `None ASDS` = no landing attempt

In [6]:
landing_outcomes = df["Outcome"].value_counts()
landing_outcomes

Outcome
True ASDS      41
None None      19
True RTLS      14
False ASDS      6
True Ocean      5
False Ocean     2
None ASDS       2
False RTLS      1
Name: count, dtype: int64

In [7]:
# bad outcomes = every outcome that does not start with "True"
bad_outcomes = {o for o in landing_outcomes.index if not o.startswith("True")}
bad_outcomes

{'False ASDS', 'False Ocean', 'False RTLS', 'None ASDS', 'None None'}

## 5. Create the label `Class`

In [8]:
df["Class"] = np.where(df["Outcome"].isin(bad_outcomes), 0, 1)
df[["FlightNumber", "Outcome", "Class"]].head(8)

,FlightNumber,Outcome,Class
0,1,None None,0
1,2,None None,0
2,3,None None,0
3,4,False Ocean,0
4,5,None None,0
5,6,None None,0
6,7,True Ocean,1
7,8,True Ocean,1


In [9]:
print("Successful landings (Class=1):", df["Class"].sum(), "of", len(df))
print("Overall success rate:", round(df["Class"].mean() * 100, 1), "%")

Successful landings (Class=1): 60 of 90
Overall success rate: 66.7 %


## 6. Save `dataset_part_2.csv`

In [10]:
df.to_csv("../data/dataset_part_2.csv", index=False)
print("saved ../data/dataset_part_2.csv", df.shape)

saved ../data/dataset_part_2.csv (90, 18)


## 7. Feature engineering → `dataset_part_3.csv`
We choose the features used for prediction and turn the text columns into 0/1 columns (one-hot encoding).

In [11]:
features = df[["FlightNumber", "PayloadMass", "Orbit", "LaunchSite", "Flights", "GridFins",
               "Reused", "Legs", "LandingPad", "Block", "ReusedCount", "Serial"]]
features_one_hot = pd.get_dummies(features, columns=["Orbit", "LaunchSite", "LandingPad", "Serial"])
features_one_hot = features_one_hot.astype("float64")      # every column becomes a number
print(features_one_hot.shape)
features_one_hot.head()

(90, 80)


,FlightNumber,PayloadMass,Flights,GridFins,Reused,Legs,Block,ReusedCount,Orbit_ES-L1,Orbit_GEO,...,Serial_B1048,Serial_B1049,Serial_B1050,Serial_B1051,Serial_B1054,Serial_B1056,Serial_B1058,Serial_B1059,Serial_B1060,Serial_B1062
0,1.0,6123.547647,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2.0,525.000000,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,3.0,677.000000,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,4.0,500.000000,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,5.0,3170.000000,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [12]:
features_one_hot.to_csv("../data/dataset_part_3.csv", index=False)
print("saved ../data/dataset_part_3.csv", features_one_hot.shape)

saved ../data/dataset_part_3.csv (90, 80)
